# Librerías

# Ingesta SBS Librería

**Objetivo**: scrapear listados de productos desde sbs.com.pe, guardar un snapshot crudo + una versión deduplicada en y subir a HDFS.

**Alcance**:

- Descubrimiento de categorías.

- Crawl de listados con paginación.

- Extracción: título, precio actual, precio anterior, URL y categoría.

- Guardado CSV crudo + CSV deduplicado (por url_norm).

## Dependencias

In [1]:
# %pip install --quiet requests beautifulsoup4 lxml pandas numpy

# Parámetros

In [2]:
from datetime import datetime, date
import os, time, random

print("=== INGESTA SBS — ONE SHOT ===")
print("Inicio:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

RAW_DAY   = date.today().isoformat()
LOCAL_DIR = "/srv/bigdata"          # staging local
os.makedirs(LOCAL_DIR, exist_ok=True)

print(f"RAW_DAY  : {RAW_DAY}")
print(f"LOCAL_DIR: {LOCAL_DIR}")

=== INGESTA SBS — ONE SHOT ===
Inicio: 2025-09-30 11:41:19
RAW_DAY  : 2025-09-30
LOCAL_DIR: /srv/bigdata


## Conectividad

In [3]:
import requests

BASE = "https://www.sbs.com.pe"
USER_AGENT = ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
              "AppleWebKit/537.36 (KHTML, like Gecko) "
              "Chrome/120.0.0.0 Safari/537.36")
HEADERS = {"User-Agent": USER_AGENT}
REQUEST_TIMEOUT = 20

print("Chequeando HOME…", BASE, end="  ")
r = requests.get(BASE, headers=HEADERS, timeout=REQUEST_TIMEOUT, allow_redirects=True)
print("OK" if r.ok else f"FALLA ({r.status_code})")


Chequeando HOME… https://www.sbs.com.pe  OK


## Utilidades

In [4]:
import re
from urllib.parse import urljoin, urlparse
from bs4 import BeautifulSoup

def clean_money(s: str | None):
    if not s:
        return None
    s = (s.replace("\u00a0"," ")
           .replace("S/.", "")
           .replace("S/", "")
           .replace("S/ ", "")
           .strip())
    digits = "".join(ch for ch in s if ch.isdigit() or ch in ".,")
    digits = digits.replace(",", "")
    try:
        return float(digits)
    except:
        return None

def fetch(url: str) -> str:
    r = requests.get(url, headers=HEADERS, timeout=REQUEST_TIMEOUT)
    r.raise_for_status()
    return r.text

# selectores flexibles (Magento-like)
SELECTORS = {
    "product_card": [
        "li.product", "li.product-item", "div.product-item",
        "div.product.product-item", "ol.products li", "div.products div.product"
    ],
    "title": [
        "a.product-item-link", "h2.product.name a", "h3.product-title a", "a.product.name"
    ],
    "price_current": [
        "span.price-final_price span.price", "span.special-price span.price",
        "span.price-wrapper span.price", "span.price"
    ],
    "price_old": [
        "span.old-price span.price", "del span.price",
        "span.price-wrapper .old-price span.price", "del.price"
    ],
    "link": [
        "a.product-item-link", "h2.product.name a", "a.product.name", "h3.product-title a"
    ],
    "next_page": [
        "a[title='Siguiente']", "a.next", "a[rel='next']",
        "li.pages-item-next a", "a.action.next"
    ],
}

def first_text(el, candidates):
    for sel in candidates:
        f = el.select_one(sel)
        if f and f.get_text(strip=True):
            return f.get_text(strip=True)
    return ""

def first_attr(el, candidates, attr="href"):
    for sel in candidates:
        f = el.select_one(sel)
        if f and f.get(attr):
            return f.get(attr)
    return ""

def url_to_catname(u: str) -> str:
    p = urlparse(u).path.strip("/").replace(".html","")
    parts = [x for x in p.split("/") if x]
    return " / ".join([x.replace("-", " ").title() for x in parts[:4]]) or "General"


## Taxonomía

In [5]:
taxonomy = {
    "Accesorios de Lectura": ["Marcapáginas", "Bolsos de Tela", "Cartucheras y Accesorios"],
    "Desarrollo Personal y Bienestar": {
        "Desarrollo Personal": ["Autoestima", "Autoayuda", "Espiritualidad y Orientalismo",
                                "Inteligencia Emocional", "Mandalas y Libros para Colorear",
                                "Meditación", "Mindfulness"],
        "Salud y Bienestar": {
            "Ejercicios y Vida Saludable": ["Alimentación Saludable", "Yoga"],
            "Medicina Alternativa": [],
            "Salud Mental": []
        },
        "Espiritualidad y Orientalismo": []
    },
    "Magia y Esoterismo": ["Ángeles", "Astrología", "Enigmas y Conspiraciones", "Sueños",
                           "Tarots", "Magia", "Fenómenos Paranormales y Parapsicología"],
    "Plan Lector": ["Inicial", "Primaria", "Secundaria"],
    "No Ficción": {
        "Ensayo y Política Peruana": [],
        "Vida y Hogar": ["Maternidad y Embarazo", "Hogar, Manualidades y Jardinería",
                         "Crianza de los Hijos. Parenting"],
        "Actualidad": [],
        "Libros de Gastronomía": ["Cocina Peruana", "Bebidas y Licores", "Cocina Internacional",
                                  "Vegetariana y Vegana", "Postres"],
        "Arte, Arquitectura y Fotografía": [],
        "Biografías": [],
        "Historia": ["Historia del Perú", "Historial Universal"],
        "Humanidades": ["Ciencias Sociales", "Filosofía"],
        "Deportes y Recreación": [],
        "Viajes y Turismo": ["Guías Turísticas", "Ilustrados", "Perú"]
    },
    "Empresa y Management": ["Liderazgo", "Management", "Educación Financiera", "Éxito Empresarial",
                             "Gestión del Tiempo y Productividad", "Inteligencia Emocional",
                             "Programación Neurolingüistica (PNL)"],
    "Pasatiempos y Rompecabezas": ["Pasatiempos", "Juegos de Mesa", "Rompecabezas"],
    "Idiomas": {
        "Alemán": ["Aprender Alemán", "Diccionarios", "Gramática y vocabulario"],
        "Francés": ["Aprender Francés", "Diccionarios", "Gramática y vocabulario"],
        "Inglés": {
            "Aprender Inglés": {
                "Básico": ["Niños", "Jóvenes", "Adultos"],
                "Intermedio": ["Niños", "Jóvenes", "Adultos"],
                "Avanzado": ["Jóvenes", "Adultos"]
            },
            "Diccionarios": ["Monolingue", "Biligue"],
            "Exámenes Internacionales": ["IELTS", "GRE", "TKT", "TOEFL", "KET", "PET", "FCE",
                                         "CAE", "CPE", "GMAT", "SAT"],
            "Gramatica y Vocabulario": ["Habilidades de Idiomas, Skills"],
            "Readers y Lecturas Graduadas": [],
            "Editoriales": ["Cambridge University Press", "Pearson Education",
                            "Oxford University Press", "National Geografic Learning", "Collins"]
        },
        "Italiano": ["Aprender Italiano", "Diccionarios"],
        "Portugués": ["Aprender Portugués", "Diccionarios", "Gramatica y Vocabulario"],
        "Otros Idiomas": ["Chino", "Español", "Quechua", "Japonés", "Ruso"]
    },
    "Juvenil": {
        "Literatura y ficción": {
            "Fantasía": ["Harry Potter", "Percy Jackson"],
            "Misterio, Suspenso y Terror": [],
            "Romance": []
        },
        "Videojuegos, Youtubers e Influencers": []
    },
    "Libros para Niños": {
        "De 0 a 2 años": ["Libros con Texturas", "Libros para la Hora del Baño",
                          "Primeros Conocimientos", "Cuentos (Libros de Cartón)"],
        "De 3 a 5 años": ["Animales", "Cuentos", "Cuentos con Valores", "Dinosaurios",
                          "Libros Interactivos", "Libros Didácticos y Educativos",
                          "Libros Montessori", "Libros de Emociones", "Prelectura y Escritura",
                          "Colección de Cuentos", "Cuentos Clásicos"],
        "De 6 a 9 años": ["Dinosaurios", "Colección de Cuentos", "Cuentos", "Cuentos con Valores",
                          "Libros de Emociones", "Libros de Actividades", "Ciencias y Naturaleza",
                          "Cuerpo Humano", "Historia", "Biblia para Niños",
                          "Enciclopedias Infantiles", "Libros Didácticos y Educativos"],
        "Literatura Infantil": ["Leyendas, Fábulas y Mitos", "Cuentos"],
        "Sagas Infantiles": ["Harry Potter", "Otras Sagas", "Percy Jackson", "Diario de Greg",
                             "Diario de Nikki", "Hombre Perro", "Isadora Moon", "Dog Man",
                             "Wimpy Kid", "Star Wars"],
        "Libros en Inglés para niños": []
    },
    "Literatura y Ficción": ["Box Sets y Colecciones", "Novela Histórica",
                             "Misterio, Thriller, Terror y Suspenso", "Poesía", "Literatura Clásica",
                             "Literatura en Inglés", "Literatura Peruana",
                             "Narrativa Contemporánea", "Narrativa Romántica",
                             "Literatura en Otros Idiomas", "Cuentos", "Fantasía y Ciencia ficción"],
    "Profesional y Técnico": {
        "Administración": [], "Contabilidad": [], "Economía": [], "Finanzas": [],
        "Recursos Humanos": [], "Marketing y Ventas": [],
        "Libros Técnicos": ["Arquitectura", "Ecología y Medio Ambiente", "Química", "Psicología",
                            "Comunicación", "Derecho y Ciencias Politicas", "Ingeniería",
                            "Logística", "Negocios Internacionales", "Otras Ciencias",
                            "Física", "Medicina"],
        "Educación y Referencia": ["Diccionarios y Enciclopedias",
                                   "Metodología de Investigación", "Lenguaje y Gramática"],
        "Metodología de Investigación": [],
        "Fondos Universitarios": ["Fondo Editorial Universidad del Pacífico",
                                  "Fondo Editorial UPC"]
    },
    "Cómics, Manga y Novelas Gráficas": {
        "Comics": ["Cómic Independiente", "DC Comics", "Marvel"],
        "Manga": [],
        "Novelas Gráficas": []
    },
    "Paperblanks": ["Agendas", "Rompecabezas", "Libretas", "Bolsos de Tela",
                    "Marcapáginas", "Estuches y Accesorios"],
    "Especiales": ["Súper Ofertas", "Novedades", "Diccionarios",
                   "FIL Online Penguin Random House", "Estuches Paperblanks",
                   "Libros en Inglés 30", "Festival de Importados"]
}

In [6]:
import unicodedata

def slugify(name: str) -> str:
    s = unicodedata.normalize("NFKD", name).encode("ascii","ignore").decode("ascii")
    s = s.lower()
    s = s.replace("&"," y ").replace("/", " ").replace(".", " ")
    s = re.sub(r"\s+y\s+"," y ", s)
    s = re.sub(r"[^a-z0-9\s-]","", s)
    s = re.sub(r"\s+","-", s).strip("-")
    return s

def paths_from_taxonomy(tree, prefix=""):
    paths = set()
    if isinstance(tree, dict):
        for k, v in tree.items():
            p = "/".join(x for x in [prefix, slugify(k)] if x)
            paths.add(p)
            paths |= paths_from_taxonomy(v, p)
    elif isinstance(tree, list):
        for k in tree:
            p = "/".join(x for x in [prefix, slugify(k)] if x)
            paths.add(p)
    return paths

def check_url(url):
    try:
        r = requests.get(url, headers=HEADERS, timeout=REQUEST_TIMEOUT, allow_redirects=True)
        return r.status_code == 200
    except:
        return False

candidate_paths = sorted(paths_from_taxonomy(taxonomy))
CATEGORIES = []
for p in candidate_paths:
    u = f"{BASE}/{p}.html"
    if check_url(u):
        CATEGORIES.append(u)

print("=== Categorías validadas ===")
print("Total:", len(CATEGORIES))
for u in CATEGORIES[:10]:
    print(" -", u)

assert CATEGORIES, "No se validó ninguna categoría; revisa conectividad/slug/estructura del sitio."


=== Categorías validadas ===
Total: 210
 - https://www.sbs.com.pe/accesorios-de-lectura.html
 - https://www.sbs.com.pe/accesorios-de-lectura/bolsos-de-tela.html
 - https://www.sbs.com.pe/accesorios-de-lectura/cartucheras-y-accesorios.html
 - https://www.sbs.com.pe/accesorios-de-lectura/marcapaginas.html
 - https://www.sbs.com.pe/comics-manga-y-novelas-graficas.html
 - https://www.sbs.com.pe/comics-manga-y-novelas-graficas/comics.html
 - https://www.sbs.com.pe/comics-manga-y-novelas-graficas/comics/comic-independiente.html
 - https://www.sbs.com.pe/comics-manga-y-novelas-graficas/comics/dc-comics.html
 - https://www.sbs.com.pe/comics-manga-y-novelas-graficas/comics/marvel.html
 - https://www.sbs.com.pe/comics-manga-y-novelas-graficas/manga.html
